# Lab 2: Personal Task Tracker

**Difficulty: Beginner | ~40 min | No prerequisites**

In this lab, you will build a personal task tracker using MongoDB.

You will learn how to:
1. Connect to a MongoDB instance
2. Insert tasks with `insert_one()` and `insert_many()`
3. Update task state with `update_one()` and `update_many()`
4. Use upserts to add-if-missing-or-update
5. Delete single and multiple tasks
6. Re-query after every mutation to see data change in real time

In [ ]:
!pip install -qU pymongo==4.10.1 mongomock

This installs the MongoDB Python driver (`pymongo`) and an in-memory mock server (`mongomock`) so you can practice without installing MongoDB on your machine.

### Step 1 — Connect to MongoDB

In [ ]:
import pymongo
import mongomock

# Create an in-memory MongoDB client (no server needed)
# To use a real MongoDB server, replace with: client = pymongo.MongoClient("mongodb://localhost:27017/")
client = mongomock.MongoClient()

# Access (or create) the database and collection
db = client["task_db"]
tasks = db["tasks"]

print("Connected to MongoDB (in-memory mock)")

We create a `mongomock.MongoClient()` which behaves exactly like a real MongoDB connection but runs in memory. To switch to a real server later, you only change this one line.

### Step 2 — Insert Tasks (insert_one + insert_many)

In [ ]:
first_task = {
    "title": "Design login page",
    "description": "Create wireframes for the new login flow",
    "priority": "critical",
    "status": "pending",
    "due_date": "2026-01-20",
    "category": "work"
}

result = tasks.insert_one(first_task)
print(f"Inserted 1 task with insert_one().")

remaining_tasks = [
    {"title": "Write unit tests",       "description": "Add tests for auth module",       "priority": "high",   "status": "pending", "due_date": "2026-01-25", "category": "work"},
    {"title": "Fix navigation bug",     "description": "Menu not closing on mobile",       "priority": "critical","status": "pending", "due_date": "2026-01-22", "category": "work"},
    {"title": "Deploy staging server",  "description": "Push latest build to staging",     "priority": "medium", "status": "pending", "due_date": "2026-02-01", "category": "work"},
    {"title": "Learn Docker basics",    "description": "Complete Docker tutorial series",  "priority": "low",    "status": "pending", "due_date": "2026-02-10", "category": "learning"},
    {"title": "Review pull requests",   "description": "Review 3 open PRs on GitHub",      "priority": "high",   "status": "pending", "due_date": "2026-01-28", "category": "work"},
    {"title": "Update documentation",   "description": "Refresh API docs with new endpoints","priority": "medium","status": "pending", "due_date": "2026-02-05", "category": "work"},
    {"title": "Plan team offsite",      "description": "Book venue and send invites",      "priority": "low",    "status": "pending", "due_date": "2026-02-15", "category": "personal"},
]

result = tasks.insert_many(remaining_tasks)
print(f"Inserted {len(result.inserted_ids)} tasks with insert_many().")
print(f"Total tasks in collection: {tasks.count_documents({})}")

`insert_one()` adds a single document. `insert_many()` accepts a list of dictionaries and inserts them all in one call — much faster than calling `insert_one()` in a loop. Unlike SQL, no table creation is needed; MongoDB creates the collection automatically on first insert.

### Step 3 — Query: View All Tasks

In [ ]:
print("--- All Tasks (after insert) ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10} | {t['due_date']}")

The empty filter `{}` matches all documents. The `{"_id": 0}` projection excludes MongoDB's auto-generated `_id` field, keeping output clean.

### Step 4 — Query: Tasks Due Soon

In [ ]:
print("--- Tasks Due By End of January 2026 ---")
for t in tasks.find(
    {"due_date": {"$lte": "2026-01-31"}, "status": {"$ne": "completed"}},
    {"_id": 0}
):
    print(f"{t['title']:<25} | Due: {t['due_date']} | Priority: {t['priority']}")

The `$lte` operator means "less than or equal to." Combining it with `{"$ne": "completed"}` filters out tasks that are already done. String-based date comparisons work here because the dates are in ISO 8601 format (`YYYY-MM-DD`), which sorts lexicographically.

### Step 5 — Update: Mark a Task as Completed (update_one)

In [ ]:
result = tasks.update_one(
    {"title": "Design login page"},
    {"$set": {"status": "completed", "completed_date": "2026-01-19"}}
)
print(f"Matched {result.matched_count}, modified {result.modified_count}")

print("\n--- All Tasks (after marking complete) ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10} | {t['due_date']}")

`update_one()` modifies the **first** document that matches the filter. The `$set` operator updates only the specified fields — `status` becomes `"completed"` and a new `completed_date` field is added. All other fields remain unchanged.

### Step 6 — Update: Bump Priority for All High-Priority Tasks (update_many)

In [ ]:
result = tasks.update_many(
    {"priority": "high", "status": {"$ne": "completed"}},
    {"$set": {"priority": "critical"}}
)
print(f"Matched {result.matched_count}, modified {result.modified_count}")

print("\n--- All Tasks (after priority bump) ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10} | {t['due_date']}")

`update_many()` modifies **all** documents that match the filter. Here, every task with `"priority": "high"` that isn't completed gets bumped to `"critical"`. The `modified_count` tells you exactly how many documents changed.

### Step 7 — Upsert: Add a Task if Missing, Else Update (update_one with upsert)

In [ ]:
result = tasks.update_one(
    {"title": "Update user profile"},
    {"$set": {"title": "Update user profile", "description": "Add profile photo",
              "priority": "medium", "status": "pending", "due_date": "2026-02-20",
              "category": "personal"}},
    upsert=True
)
print(f"Matched {result.matched_count}, modified {result.modified_count}, upserted_id {result.upserted_id}")

print("\n--- All Tasks (after upsert) ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10} | {t['due_date']}")

With `upsert=True`, MongoDB first tries to find a document matching the filter. If found, it updates it. If not found, it **inserts** a new document with the fields from the update. Since "Update user profile" doesn't exist yet, this creates it. Running the same cell again would update the existing document instead of creating a duplicate.

### Step 8 — Delete: Remove a Single Task (delete_one)

In [ ]:
result = tasks.delete_one({"title": "Learn Docker basics"})
print(f"Deleted {result.deleted_count} task: Learn Docker basics")

print("\n--- Remaining Tasks ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10}")
print(f"\nTotal tasks: {tasks.count_documents({})}")

`delete_one()` removes the **first** document matching the filter. Only one document is deleted, even if multiple match.

### Step 9 — Delete: Remove All Completed Tasks (delete_many)

In [ ]:
result = tasks.delete_many({"status": "completed"})
print(f"Deleted {result.deleted_count} completed tasks.")

print("\n--- Remaining Active Tasks ---")
for t in tasks.find({}, {"_id": 0}):
    print(f"{t['title']:<25} | {t['priority']:<8} | {t['status']:<10} | Due: {t['due_date']}")
print(f"\nTotal tasks: {tasks.count_documents({})}")

`delete_many()` removes **all** documents matching the filter. This is how you'd archive or clean up finished work — query for completed items, then delete them in one call.

### Step 10 — Print Summary Report

In [ ]:
all_tasks = list(tasks.find({}, {"_id": 0}))
high_priority = [t for t in all_tasks if t["priority"] in ("critical", "high")]
overdue = [t for t in all_tasks if t["due_date"] < "2026-01-31" and t["status"] != "completed"]

print("         PERSONAL TASK TRACKER — SUMMARY")
print(f"\nTotal active tasks: {len(all_tasks)}")
print(f"Critical/high priority: {len(high_priority)}")
print(f"Overdue (due before Feb 1): {len(overdue)}")

print("\n--- High Priority Tasks ---")
for t in high_priority:
    print(f"  [{t['priority'].upper():<8}] {t['title']} — Due: {t['due_date']}")

print("\n--- Overdue Tasks ---")
for t in overdue:
    print(f"  {t['title']} — Due: {t['due_date']} (still {t['status']})")

print("\n--- All Active Tasks by Due Date ---")
for t in sorted(all_tasks, key=lambda x: x["due_date"]):
    print(f"  {t['due_date']} | {t['title']:<25} | {t['priority']:<8} | {t['status']}")

This final cell collects all remaining data into a formatted summary — the kind of dashboard a project manager would actually use to plan their day.